# RQ1.2 — двухстадийная: добор и переобучение (Colab)

Три задачи в одном прогоне:

1. **Переобучение всей двухстадийной ветки U-DIADS-TL по актуальному рецепту.**
   Существующие 27 ячеек сделаны при накоплении 16 и cosine; скрипт с тех пор
   переведён на накопление 2 и constant (`# was 16 -> 5x more optimizer steps`).
   Это, вероятно, и объясняет слабость детекторов U-DIADS: медиана test_mAP50
   0.200-0.422 против 0.75-0.90 у DIVA.
2. **`pvt_v2_b2` для U-DIADS** — размер M в ветке `pvt`, локально не влезает.
3. **Семейство `vit` через SFP-шею** — плоский ViT в RT-DETR напрямую не идёт
   (все уровни выходят с одним страйдом), но `hier_encoder` из `71_misc/`
   надстраивает над ним ViTDet-подобную пирамиду и даёт настоящие 8/16/32.
   Поддержка добавлена в `train_rtdetr_hf.py` (`_SFP_BACKBONES`, `_build_sfp_model`).
   Веса берутся через timm: прямая загрузка dinov3 от Meta отдаёт 403 и **молча**
   подставляет случайный ViT.

   Оговорка для текста работы: у `convnext` и `pvt` пирамида своя, у `vit` она
   достраивается обучаемой шеей. Это сравнение «ViT+SFP против CNN», а не
   «ViT против CNN» — так и надо называть.
4. **`imagenet` для `convnext_femto` и `convnext_pico`** на обоих датасетах.
   HF не публикует ConvNeXt меньше `tiny`, поэтому веса берутся из timm и
   пересаживаются переименованием ключей — поддержка добавлена в
   `train_rtdetr_hf.py` (`_TIMM_PRETRAINED`, `_timm_convnext_to_hf`). Тело
   бэкбона после пересадки совпадает с timm с точностью 1e-4.

Рецепт единый для всего: 100 эпох, lr 1e-4, батч 1, накопление 2, constant,
seed 42, bf16, 1152 px — тот же, что у всех арок DIVA. После прогона обе
таблицы становятся однородными.

Старые веса U-DIADS **перезаписываются**: имена прогонов те же. Если нужны
прежние числа для сравнения, сохраните `99_evaluation/02_2stage/u-diads-tl/`
до запуска.

---

**Чего этот ноутбук НЕ делает.** Оставшиеся 12 ячеек RQ1.2 — `imagenet` для
`convnext_femto` и `convnext_pico` на обоих датасетах — заблокированы: веса
ImageNet для этих размеров публикуются только в формате timm
(`timm/convnext_femto.d1_in1k`), а `train_rtdetr_hf.py` грузит бэкбон через
`AutoBackbone`. Прямой перенос не проходит — у timm схема имён `stem_0.weight`,
у HF `embeddings.patch_embeddings.weight`, совпадают 14 форм из 22. Нужен слой
переименования ключей, его в коде нет.

## 1. Конфигурация

In [ ]:

import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = Path("/content/drive/MyDrive/Thesis")
REPO_DIR    = Path("/content/repo")
WORK_DIR    = Path("/content/work")

DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
DRIVE_OUT    = DRIVE_ROOT / "99_evaluation"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"

UDIADS = ["Latin14396", "Latin2", "Syr341"]
DIVA   = ["CB55", "CS18", "CS863"]

# (backbone, init) pairs. DINOv3 exists only for convnext_tiny; PvT has none.
# (backbone, init). DINOv3 exists only for convnext_tiny and vit_small -- the
# smallest DINO weights of any family are 21.6M, so XS/S cannot have that column.
CONV_PVT = [("convnext_femto", "random"), ("convnext_femto", "imagenet"),
            ("convnext_pico",  "random"), ("convnext_pico",  "imagenet"),
            ("convnext_tiny",  "random"), ("convnext_tiny",  "imagenet"),
            ("convnext_tiny",  "dinov3"),
            ("pvt_v2_b0", "random"), ("pvt_v2_b0", "imagenet"),
            ("pvt_v2_b1", "random"), ("pvt_v2_b1", "imagenet"),
            ("pvt_v2_b2", "random"), ("pvt_v2_b2", "imagenet")]
# Flat ViTs, hierarchical only through the SFP neck.
VIT = [("vit_tiny",  "random"), ("vit_tiny",  "imagenet"),
       ("vit_small", "random"), ("vit_small", "imagenet"), ("vit_small", "dinov3")]

FULL_GRID = CONV_PVT + VIT          # U-DIADS: everything, old recipe is superseded
DIVA_GRID = [("convnext_femto", "imagenet"), ("convnext_pico", "imagenet")] + VIT

# The recipe lives inside train_rtdetr_hf.py (100 epochs, lr 1e-4, batch 1,
# accum 2, constant, seed 42) and is NOT overridable by environment variables --
# only these five are read: DATASET, BACKBONE, INIT, IMAGE_SIZE, RUN_NAME.
IMAGE_SIZE   = 1152
TRANSFORMERS_PIN = "5.14.1"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def sh(cmd, check=True, cwd=None, env=None):
    print("$", cmd)
    full = {**os.environ, **(env or {})}
    return subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=full)


ARMS = ([("U-DIADS-TL", s, b, i) for s in UDIADS for b, i in FULL_GRID]
        + [("DIVA-HisDB", s, b, i) for s in DIVA for b, i in DIVA_GRID])
print(f"к обучению: {len(ARMS)} детекторов")
print(f"   U-DIADS-TL: {len(UDIADS) * len(FULL_GRID)} (полное переобучение + vit)")
print(f"   DIVA-HisDB: {len(DIVA) * len(DIVA_GRID)} (imagenet для femto/pico + vit)")

## 2. Диск

In [ ]:

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} не найден — создайте его на Диске"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} не найден — залейте туда 00_data"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print("диск в порядке:", DRIVE_ROOT)

## 3. Репозиторий

Используются только отслеживаемые файлы `50_modelling/02_2stage/`.
`00_data` и `80_models` подставляются симлинками на локальный SSD — обучение
с FUSE-монтирования Диска идёт в разы медленнее.

In [ ]:

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS):
    d.mkdir(parents=True, exist_ok=True)

for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target)

STAGE_DIR = REPO_DIR / "50_modelling" / "02_2stage"
assert (STAGE_DIR / "train_rtdetr_hf.py").exists(), "train_rtdetr_hf.py не найден"
print("репозиторий готов:", REPO_DIR)

## 4. Данные

Детектору нужны COCO-разметка и изображения; оценка добавляет исходные
страницы и разметку U-DIADS.

In [ ]:

NEEDED = []
for s in UDIADS:
    NEEDED += [f"U-DIADS-TL/coco_dataset_{s}", f"U-DIADS-TL/yolo_dataset_{s}/images"]
for s in DIVA:
    NEEDED += [f"DIVA-HisDB/coco_dataset_{s}", f"DIVA-HisDB/yolo_dataset_{s}/images"]

missing = []
for rel in NEEDED:
    src, dst = DRIVE_DATA / rel, LOCAL_DATA / rel
    if dst.is_dir():
        continue
    if not src.is_dir():
        missing.append(rel)
        continue
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    print(f"[copy] {rel}")

assert not missing, f"нет на Диске: {missing}"
print("данные на месте")

## 5. Зависимости

In [ ]:

sh(f'pip -q install "transformers=={TRANSFORMERS_PIN}" torchmetrics '
   f'albumentations pycocotools datasets timm')
sh("nvidia-smi || true", check=False)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert transformers.__version__.startswith(TRANSFORMERS_PIN.split(".")[0]), \
    "мажорная версия transformers отличается от пина — перезапустите среду"

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"{props.total_memory / 1e9:.0f} ГБ")
BF16 = torch.cuda.is_bf16_supported()
print("bf16:", BF16, "(на T4 False — упадёт в fp16)")

## 6. Проверка памяти

Реальный forward+backward на b2 при 1152 px. Если не влезает — уменьшите
`IMAGE_SIZE`, но тогда ячейка станет несопоставимой с остальными арками
U-DIADS, и это надо будет отметить в таблице.

In [ ]:

import gc, torch
from transformers import AutoConfig, RTDetrConfig, AutoModelForObjectDetection

bc = AutoConfig.from_pretrained("OpenGVLab/pvt_v2_b2")   # heaviest arm in the grid
bc.out_indices = [1, 2, 3]          # stage2/3/4 -> strides 8/16/32
cfg = RTDetrConfig.from_pretrained(
    "PekingU/rtdetr_r50vd", backbone_config=bc, use_timm_backbone=False,
    num_feature_levels=3, num_labels=1,
    id2label={0: "TextLine"}, label2id={"TextLine": 0})
m = AutoModelForObjectDetection.from_config(cfg).cuda().train()

try:
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device="cuda")
    with torch.autocast("cuda", dtype=torch.bfloat16 if BF16 else torch.float16):
        loss = m(pixel_values=x, labels=[{"class_labels": torch.zeros(1, dtype=torch.long, device="cuda"),
                                          "boxes": torch.tensor([[.5, .5, .2, .05]], device="cuda")}]).loss
    loss.backward()
    print(f"OK при {IMAGE_SIZE}px: пик {torch.cuda.max_memory_allocated()/1e9:.1f} ГБ")
except torch.cuda.OutOfMemoryError:
    print(f"НЕ ВЛЕЗАЕТ при {IMAGE_SIZE}px — нужна карта побольше")
finally:
    del m
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

## 7. Обучение

Вызывается репозиторный `train_rtdetr_hf.py` через переменные окружения —
конфиг не дублируется, поэтому рецепт гарантированно тот же. Каждая готовая
арка сразу уезжает на Диск.

In [ ]:

def rel_for(dataset, subset):
    """Where train_rtdetr_hf.py writes -- CB55 keeps the flat legacy layout."""
    if dataset == "DIVA-HisDB":
        base = "02_2stage/diva-hisdb/detection/rtdetr_hf"
        return base if subset == "CB55" else f"{base}/{subset}"
    return f"02_2stage/u-diads-tl/detection/rtdetr_hf/{subset}"


def push(rel, name):
    src = LOCAL_MODELS / rel / name
    if src.is_dir():
        shutil.copytree(src, DRIVE_MODELS / rel / name, dirs_exist_ok=True)
        print(f"[push] {rel}/{name}")


for i, (dataset, subset, backbone, init) in enumerate(ARMS, 1):
    name = f"rtdetr_{backbone}_{init}"
    rel = rel_for(dataset, subset)
    done = LOCAL_MODELS / rel / name / "best_model" / "config.json"
    if done.exists():
        print(f"[skip] ({i}/{len(ARMS)}) {subset}/{name}")
        continue

    print(f"\n######## ({i}/{len(ARMS)}) {dataset} {subset} / {name} ########")
    # Only these five are read by the script; the recipe itself is hardcoded there.
    env = dict(DATASET=subset, BACKBONE=backbone, INIT=init,
               IMAGE_SIZE=str(IMAGE_SIZE), RUN_NAME=name)
    r = sh("python train_rtdetr_hf.py", check=False, cwd=STAGE_DIR, env=env)
    if r.returncode != 0:
        print(f"[FAILED] {subset}/{name} — остальные продолжают")
        continue
    push(rel, name)

## 8. Выгрузка результатов

Детекторные метрики пишутся в `results.csv` каждого подмножества. Линейные
метрики считаются локально — сегментер кропов и Java-оценщик уже настроены
там, тащить их в Colab незачем.

In [ ]:

for subset in UDIADS:
    src = REPO_DIR / f"99_evaluation/02_2stage/u-diads-tl/rtdetr_hf/{subset}/results.csv"
    if src.exists():
        dst = DRIVE_OUT / f"02_2stage/u-diads-tl/rtdetr_hf/{subset}/results.csv"
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"[push] {subset}/results.csv")

import pandas as pd
for subset in UDIADS:
    p = REPO_DIR / f"99_evaluation/02_2stage/u-diads-tl/rtdetr_hf/{subset}/results.csv"
    if p.exists():
        d = pd.read_csv(p)
        print(f"\n{subset}: {len(d)} арок, медиана test_mAP50 {d.test_mAP50.median():.3f}")
        print(d[["platform", "test_mAP50"]].sort_values("test_mAP50", ascending=False)
              .head(5).to_string(index=False))

## 9. Дальше — локально

1. Скачайте с Диска `80_models/02_2stage/` в локальный репозиторий
   (U-DIADS перезаписывается целиком, DIVA — только две новые арки).
2. Посчитайте линейные метрики тем же путём, что и остальные арки U-DIADS.
3. Пересоберите таблицы: `python 99_evaluation/make_rq1_split_tables.py`.